# 03 — Design and evaluate chunk boundaries

**Track:** Beginner · **Stage:** Ingestion & Chunking

Chunking is not clerical preprocessing. Chunking determines the unit that can be indexed and retrieved, and different chunking strategies produce measurably different retrieval behavior. A bad boundary can make the right answer unretrievable even when the corpus contains it. 

In this lab, you will compare standard splitters using **LangChain** and measure how chunk boundaries affect retrieval context and correctness.

## What you will build

- A retrieval pipeline using local embeddings (HuggingFace) and vector database (Chroma).
- A comparison between a fixed character baseline, a recursive baseline, and structure-aware splitting.
- A lightweight evaluation to measure if the correct evidence survives and ranks.
- Experiments to see how chunk size and overlap affect retrieval metrics.
- A demonstration of parent-child (small-to-big) retrieval.

## Setup: Dependencies and Stack

We will use the same local embedding stack as the previous course, alongside LangChain's chunking utilities.

In [1]:
# !pip install langchain-text-splitters langchain-chroma langchain-huggingface

import warnings
warnings.filterwarnings("ignore")

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Initialize our local embedding model (same as Course 02)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7828.03it/s]

## 1. The Corpus and Evaluation Set

We will use a small financial review. Notice how the number `14%` is used across three different sections. A poorly chunked number will lose its context, making it impossible to answer specific questions accurately.

In [2]:
raw_text = """# Q2 2025 Financial Review

## Cloud Infrastructure

The new distributed cluster was deployed in May.

Because of redundant systems and cross-region backups,
cloud infrastructure costs increased by 14%.

Uptime improved to 99.999%.

## Customer Support

Support ticket volume increased by 14% after the launch
of the new enterprise product.

## Security Operations

Critical incidents decreased by 14% after introducing
the new detection pipeline.
"""

# We will define a small evaluation set of queries
eval_queries = [
    {
        "query": "What increased by 14% because of cross-region backups?",
        "expected_section": "Cloud Infrastructure",
    },
    {
        "query": "What increased by 14% after the enterprise product launch?",
        "expected_section": "Customer Support",
    },
    {
        "query": "What improved to 99.999%?",
        "expected_section": "Cloud Infrastructure",
    },
    {
        "query": "What decreased by 14%?",
        "expected_section": "Security Operations",
    }
]

# Helper function to print chunks clearly
def print_chunks(chunks):
    for i, chunk in enumerate(chunks):
        c_id = chunk.metadata.get('chunk_id', f'chunk-{i}')
        doc_id = chunk.metadata.get('document_id', 'unknown')
        section = chunk.metadata.get('section', 'unknown')
        
        print(f"--- {c_id} [Doc: {doc_id} | Sec: {section} | Length: {len(chunk.page_content)}] ---")
        print(chunk.page_content.strip())
        print()

# Helper function to evaluate retrieval quality
def evaluate_retrieval(vectorstore, cases, k=1):
    results = []
    print(f"{'Query':<60} | {'Expected':<20} | {'Hit?'}")
    print("-" * 90)
    
    hits = 0
    for case in cases:
        docs = vectorstore.similarity_search(case["query"], k=k)
        
        # Check if the expected section appears in the retrieved documents
        hit = any(d.metadata.get("section") == case["expected_section"] for d in docs)
        if hit:
            hits += 1
            
        print(f"{case['query'][:58]:<60} | {case['expected_section']:<20} | {'✅' if hit else '❌'}")
    
    print("-" * 90)
    print(f"Recall@{k}: {hits}/{len(cases)} ({hits/len(cases)*100:.0f}%)")
    return hits


## 2. Strategy A — Fixed/Naive Character Split

This splitter blindly slices the document by character count. It does not understand paragraphs or words.

In [3]:
naive_doc = Document(page_content=raw_text, metadata={"document_id": "finance-q2-2025", "source": "q2_review.md", "section": "unknown"})

naive_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=100,
    chunk_overlap=0
)
naive_chunks = naive_splitter.split_documents([naive_doc])

# Add chunk identifiers deliberately
for i, chunk in enumerate(naive_chunks):
    chunk.metadata["chunk_id"] = f"{chunk.metadata['document_id']}#naive-{i:03d}"

print_chunks(naive_chunks)

# Evaluate
naive_db = Chroma.from_documents(naive_chunks, embeddings, collection_name="naive")
print("\nNaive Splitting Evaluation:")
evaluate_retrieval(naive_db, eval_queries, k=1)


--- finance-q2-2025#naive-000 [Doc: finance-q2-2025 | Sec: unknown | Length: 100] ---
# Q2 2025 Financial Review

## Cloud Infrastructure

The new distributed cluster was deployed in May

--- finance-q2-2025#naive-001 [Doc: finance-q2-2025 | Sec: unknown | Length: 100] ---
.

Because of redundant systems and cross-region backups,
cloud infrastructure costs increased by 14

--- finance-q2-2025#naive-002 [Doc: finance-q2-2025 | Sec: unknown | Length: 100] ---
%.

Uptime improved to 99.999%.

## Customer Support

Support ticket volume increased by 14% after t

--- finance-q2-2025#naive-003 [Doc: finance-q2-2025 | Sec: unknown | Length: 100] ---
he launch
of the new enterprise product.

## Security Operations

Critical incidents decreased by 14

--- finance-q2-2025#naive-004 [Doc: finance-q2-2025 | Sec: unknown | Length: 47] ---
% after introducing
the new detection pipeline.




Naive Splitting Evaluation:
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------
What increased by 14% because of cross-region backups?       | Cloud Infrastructure | ❌
What increased by 14% after the enterprise product launch?   | Customer Support     | ❌
What improved to 99.999%?                                    | Cloud Infrastructure | ❌
What decreased by 14%?                                       | Security Operations  | ❌
------------------------------------------------------------------------------------------
Recall@1: 0/4 (0%)


0

Notice that the naive splitter often splits words in half, and it completely loses track of what section it is in.

## 3. Strategy B — Recursive Character Split

A widely used general-purpose baseline for prose. `RecursiveCharacterTextSplitter` tries to split on paragraphs (`\n\n`), then sentences (`\n`), then words (` `), keeping natural units intact as much as possible while respecting the chunk size.

In [4]:
recursive_doc = Document(page_content=raw_text, metadata={"document_id": "finance-q2-2025", "source": "q2_review.md", "section": "unknown"})

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)
recursive_chunks = recursive_splitter.split_documents([recursive_doc])

for i, chunk in enumerate(recursive_chunks):
    chunk.metadata["chunk_id"] = f"{chunk.metadata['document_id']}#recursive-{i:03d}"

print_chunks(recursive_chunks)

# Evaluate
recursive_db = Chroma.from_documents(recursive_chunks, embeddings, collection_name="recursive")
print("\nRecursive Splitting Evaluation:")
evaluate_retrieval(recursive_db, eval_queries, k=1)


--- finance-q2-2025#recursive-000 [Doc: finance-q2-2025 | Sec: unknown | Length: 101] ---
# Q2 2025 Financial Review

## Cloud Infrastructure

The new distributed cluster was deployed in May.

--- finance-q2-2025#recursive-001 [Doc: finance-q2-2025 | Sec: unknown | Length: 128] ---
Because of redundant systems and cross-region backups,
cloud infrastructure costs increased by 14%.

Uptime improved to 99.999%.

--- finance-q2-2025#recursive-002 [Doc: finance-q2-2025 | Sec: unknown | Length: 131] ---
## Customer Support

Support ticket volume increased by 14% after the launch
of the new enterprise product.

## Security Operations

--- finance-q2-2025#recursive-003 [Doc: finance-q2-2025 | Sec: unknown | Length: 81] ---
Critical incidents decreased by 14% after introducing
the new detection pipeline.


Recursive Splitting Evaluation:
Query                                                        | Expected             | Hit?
--------------------------------------------------------------------

0

Recursive splitting preserves words and paragraphs, which improves retrieval slightly, but it still doesn't know the **Section**. If a chunk says "costs increased by 14%", it doesn't inherently contain the text "Cloud Infrastructure".

## 4. Strategy C — Markdown Heading-Aware Split

Structure-aware chunking uses the document's inherent structure (like Markdown headings) to split the text. This allows us to push the heading into the **metadata** of every child chunk beneath it, preserving context across the entire section.

In [5]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "section"), # We map ## to "section" metadata
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
markdown_chunks = markdown_splitter.split_text(raw_text)

# Add base metadata that wasn't derived from headings
for i, chunk in enumerate(markdown_chunks):
    chunk.metadata["document_id"] = "finance-q2-2025"
    chunk.metadata["source"] = "q2_review.md"
    chunk.metadata["chunk_id"] = f"{chunk.metadata['document_id']}#markdown-{i:03d}"
    # Note: 'section' is already populated by the MarkdownHeaderTextSplitter!

print_chunks(markdown_chunks)

# Evaluate
markdown_db = Chroma.from_documents(markdown_chunks, embeddings, collection_name="markdown")
print("\nMarkdown Splitting Evaluation:")
evaluate_retrieval(markdown_db, eval_queries, k=1)


--- finance-q2-2025#markdown-000 [Doc: finance-q2-2025 | Sec: Cloud Infrastructure | Length: 180] ---
The new distributed cluster was deployed in May.  
Because of redundant systems and cross-region backups,
cloud infrastructure costs increased by 14%.  
Uptime improved to 99.999%.

--- finance-q2-2025#markdown-001 [Doc: finance-q2-2025 | Sec: Customer Support | Length: 86] ---
Support ticket volume increased by 14% after the launch
of the new enterprise product.

--- finance-q2-2025#markdown-002 [Doc: finance-q2-2025 | Sec: Security Operations | Length: 81] ---
Critical incidents decreased by 14% after introducing
the new detection pipeline.


Markdown Splitting Evaluation:
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------


What increased by 14% because of cross-region backups?       | Cloud Infrastructure | ✅
What increased by 14% after the enterprise product launch?   | Customer Support     | ✅
What improved to 99.999%?                                    | Cloud Infrastructure | ❌
What decreased by 14%?                                       | Security Operations  | ✅
------------------------------------------------------------------------------------------
Recall@1: 3/4 (75%)


3

By preserving the `section` in the metadata, the chunk now carries its provenance! 

*Note: If the resulting Markdown sections were still too large (e.g. > 1000 characters), you would pass these chunks through the `RecursiveCharacterTextSplitter` afterward to bound their maximum size.*

## 5. Experiment: Varying Chunk Size

Is there a universal "best" chunk size? No. Chunk size is a tunable retrieval parameter. Let's see how changing the chunk size affects the recursive splitter's performance.

In [6]:
for size in [80, 150, 300]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)
    chunks = splitter.split_documents([recursive_doc])
    
    # ensure chunks are properly populated with section metadata to evaluate
    for chunk in chunks: chunk.metadata['section'] = 'unknown' 
    
    db = Chroma.from_documents(chunks, embeddings, collection_name=f"size_{size}")
    
    print(f"--- Chunk Size: {size} ---")
    print(f"Generated {len(chunks)} chunks. Average length: {sum(len(c.page_content) for c in chunks)/len(chunks):.0f} chars.")
    evaluate_retrieval(db, eval_queries, k=1)
    print("\n")


--- Chunk Size: 80 ---
Generated 10 chunks. Average length: 43 chars.
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------
What increased by 14% because of cross-region backups?       | Cloud Infrastructure | ❌
What increased by 14% after the enterprise product launch?   | Customer Support     | ❌


What improved to 99.999%?                                    | Cloud Infrastructure | ❌
What decreased by 14%?                                       | Security Operations  | ❌
------------------------------------------------------------------------------------------
Recall@1: 0/4 (0%)




--- Chunk Size: 150 ---
Generated 4 chunks. Average length: 110 chars.
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------
What increased by 14% because of cross-region backups?       | Cloud Infrastructure | ❌
What increased by 14% after the enterprise product launch?   | Customer Support     | ❌


What improved to 99.999%?                                    | Cloud Infrastructure | ❌
What decreased by 14%?                                       | Security Operations  | ❌
------------------------------------------------------------------------------------------
Recall@1: 0/4 (0%)


--- Chunk Size: 300 ---
Generated 2 chunks. Average length: 222 chars.
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------
What increased by 14% because of cross-region backups?       | Cloud Infrastructure | ❌


What increased by 14% after the enterprise product launch?   | Customer Support     | ❌
What improved to 99.999%?                                    | Cloud Infrastructure | ❌
What decreased by 14%?                                       | Security Operations  | ❌
------------------------------------------------------------------------------------------
Recall@1: 0/4 (0%)




Smaller chunks might lose the surrounding context (missing hits), while larger chunks might dilute the embedding with too many competing concepts. The optimal size depends on the corpus and the expected query distribution.

## 6. Experiment: Varying Overlap

Overlap can reduce some boundary failures by repeating content across adjacent chunks. Let's compare 0 overlap with 50 overlap.

In [7]:
for overlap in [0, 50]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=overlap)
    chunks = splitter.split_documents([recursive_doc])
    
    for chunk in chunks: chunk.metadata['section'] = 'unknown' 
    
    db = Chroma.from_documents(chunks, embeddings, collection_name=f"overlap_{overlap}")
    
    print(f"--- Overlap: {overlap} ---")
    print(f"Generated {len(chunks)} chunks. Average length: {sum(len(c.page_content) for c in chunks)/len(chunks):.0f} chars.")
    evaluate_retrieval(db, eval_queries, k=1)
    print("\n")


--- Overlap: 0 ---
Generated 4 chunks. Average length: 110 chars.
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------
What increased by 14% because of cross-region backups?       | Cloud Infrastructure | ❌
What increased by 14% after the enterprise product launch?   | Customer Support     | ❌
What improved to 99.999%?                                    | Cloud Infrastructure | ❌
What decreased by 14%?                                       | Security Operations  | ❌
------------------------------------------------------------------------------------------
Recall@1: 0/4 (0%)


--- Overlap: 50 ---
Generated 4 chunks. Average length: 118 chars.
Query                                                        | Expected             | Hit?
------------------------------------------------------------------------------------------
What increased by 14% because of cross-

Notice the trade-off: higher overlap might rescue a split sentence, but it generates more chunks, indexing more redundant text, and potentially causing the retriever to fetch two nearly identical chunks instead of diverse evidence.

## 7. Parent-Child (Small-to-Big) Retrieval

The unit optimized for retrieval does not always need to be the unit supplied to the generator. We can embed small chunks (for precise matching), but return their larger parent sections to the LLM (for complete context).

In [8]:
# 1. Create a dictionary mapping the Markdown sections (parents)
parent_sections = {chunk.metadata.get('section', chunk.metadata.get('Header 1', 'unknown')): chunk.page_content for chunk in markdown_chunks}

# 2. Split those sections into small child chunks for precise retrieval
child_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=0)
child_chunks = child_splitter.split_documents(markdown_chunks)

child_db = Chroma.from_documents(child_chunks, embeddings, collection_name="parent_child")

query = "What increased by 14% because of cross-region backups?"
retrieved_child = child_db.similarity_search(query, k=1)[0]
parent_section_name = retrieved_child.metadata.get('section', retrieved_child.metadata.get('Header 1', 'unknown'))

print(f"1. Retrieved Small Child Chunk [{parent_section_name}]:")
print(f"   '{retrieved_child.page_content.strip()}'\n")

print(f"2. Looked up Full Parent Context [{parent_section_name}]:")
print(f"   '{parent_sections[parent_section_name].strip()}'")


1. Retrieved Small Child Chunk [Cloud Infrastructure]:
   'Because of redundant systems and cross-region backups,'

2. Looked up Full Parent Context [Cloud Infrastructure]:
   'The new distributed cluster was deployed in May.  
Because of redundant systems and cross-region backups,
cloud infrastructure costs increased by 14%.  
Uptime improved to 99.999%.'


## 8. Failure Analysis

If you experiment with your own text and queries, always perform a failure analysis on the queries that don't retrieve the expected section. For each failure, ask:

Was the relevant information:
1. lost during parsing? (e.g. PDF table got scrambled)
2. separated by chunking? (e.g. subject is in chunk 1, metric is in chunk 2)
3. present in a chunk but poorly retrieved? (e.g. embedding model didn't understand the semantic link)
4. retrieved but surrounded by too much irrelevant text? (e.g. chunk was too large, diluting the signal)

Chunking is a design decision. Now you have the tools to measure it!